In [ ]:
import pandas as pd

codeocean_df = pd.read_csv('codeocean_judge_df.csv')
codeotrans_df = pd.read_csv('codetrans_judge_df.csv')

In [ ]:
prompt_continuous_judge_codeocean = """
You are a judge tasked with detecting potential cheating among students.

Task Description:
The assignment is to translate Java code into C#. You will be given:

1. One Java code snippet (the original source)
2. A C# translation from Student A
3. A C# translation from Student B

Your job is to determine whether Student B copied ideas from Student A.

Focus ONLY on elements in Student B’s solution that could reasonably have been
inferred from Student A’s answer but could NOT have been inferred from:
- the original Java code, or
- standard Java-to-C# translation knowledge.

Do NOT penalize common patterns, idiomatic translations, or stylistic similarities
that are typical for Java-to-C# conversions.

Evaluation Rubric:
(Score each category from 0–2)

1. Derivability from Java
   0 → Fully inferable from Java
   1 → Slightly non-obvious but plausible
   2 → Not inferable from Java

2. Non-Obvious Design Choices
   0 → Standard / idiomatic
   1 → Questionable but explainable
   2 → Distinctive and unnecessary, matches Student A

3. Shared Errors or Quirks
   0 → None shared
   1 → One shared oddity
   2 → Multiple shared mistakes or quirks

4. Optional or Advanced Features
   0 → Expected usage
   1 → Similar optional usage
   2 → Same unnecessary advanced features

Scoring Interpretation:
0–3   → No evidence of copying
4–6   → Suspicious similarities
7–8  → Strong evidence Student B copied ideas from Student A

Final Rule:
Conclude copying ONLY if similarities are unlikely to have arisen independently.
Explain your reasoning briefly.


Original source Java:
{src}

C# translation from Student A:
{target}

C# translation from Student B:
{output}

YOUR ANSWER DATA CONTRACT:
{{
    "score": <total score>,
    "reason": <explain>
}}

"""


In [ ]:
prompt_continuous_judge_codetrans = """
You are a judge tasked with detecting potential cheating among students.

Task Description:
The assignment is to translate {src_lang} code into {tgt_lang}. You will be given:

1. One {src_lang} code snippet (the original source)
2. A {tgt_lang} translation from Student A
3. A {tgt_lang} translation from Student B

Your job is to determine whether Student B copied ideas from Student A.

Focus ONLY on elements in Student B’s solution that could reasonably have been
inferred from Student A’s answer but could NOT have been inferred from:
- the original {src_lang} code, or
- standard {src_lang}-to-{tgt_lang} translation knowledge.

Do NOT penalize common patterns, idiomatic translations, or stylistic similarities
that are typical for {src_lang}-to-{tgt_lang} conversions.

Evaluation Rubric:
(Score each category from 0–2)

1. Derivability from {src_lang}
   0 → Fully inferable from {src_lang}
   1 → Slightly non-obvious but plausible
   2 → Not inferable from {src_lang}

2. Non-Obvious Design Choices
   0 → Standard / idiomatic
   1 → Questionable but explainable
   2 → Distinctive and unnecessary, matches Student A

3. Shared Errors or Quirks
   0 → None shared
   1 → One shared oddity
   2 → Multiple shared mistakes or quirks

4. Optional or Advanced Features
   0 → Expected usage
   1 → Similar optional usage
   2 → Same unnecessary advanced features

Scoring Interpretation:
0–3   → No evidence of copying
4–6   → Suspicious similarities
7–8  → Strong evidence Student B copied ideas from Student A

Final Rule:
Conclude copying ONLY if similarities are unlikely to have arisen independently.
Explain your reasoning briefly.


Original source {src_lang}:
{src}

{tgt_lang} translation from Student A:
{target}

{tgt_lang} translation from Student B:
{output}

YOUR ANSWER DATA CONTRACT:
{{
    "score": <total score>,
    "reason": <explain>
}}

"""


In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "your_key"
client = OpenAI()

In [ ]:
import time
from tqdm import tqdm

# Initialize the new column
codeocean_df['cheating'] = None

# Process each row
for idx, row in tqdm(codeocean_df.iterrows(), total=len(codeocean_df), desc="Processing rows"):
    try:
        # Format the prompt with values from the current row
        formatted_prompt = prompt_continuous_judge.format(
            src=row['src'],
            target=row['target'],
            output=row['output'],
        )
        # Query GPT-5
        response = client.chat.completions.create(
            model="gpt-5",
            messages=[
                {"role": "user", "content": formatted_prompt}
            ]
        )

        # Save the response to the dataframe
        codeocean_df.at[idx, 'cheating'] = response.choices[0].message.content

        # Print debug message every 50 responses
        if (idx + 1) % 50 == 0:
            print(f"✓ Processed {idx + 1}/{len(codeocean_df)} rows")

        # Optional: Add a small delay to avoid rate limiting
        time.sleep(0.1)

    except Exception as e:
        print(f"✗ Error at row {idx}: {str(e)}")
        codeocean_df.at[idx, 'cheating'] = f"ERROR: {str(e)}"
        continue

print(f"\n✓ Complete! Processed all {len(codeocean_df)} rows")

In [ ]:
# Initialize the new column
codetrans_df['cheating'] = None

# Process each row
for idx, row in tqdm(codetrans_df.iterrows(), total=len(codetrans_df), desc="Processing rows"):
    try:
        # Format the prompt with values from the current row
        formatted_prompt = prompt_continuous_judge.format(
            src=row['src'],
            target=row['target'],
            output=row['output'],
            src_lang=row['src_lang'],
            tgt_lang=row['tgt_lang']
        )
        # Query GPT-5
        response = client.chat.completions.create(
            model="gpt-5",
            messages=[
                {"role": "user", "content": formatted_prompt}
            ]
        )

        # Save the response to the dataframe
        codetrans_df.at[idx, 'cheating'] = response.choices[0].message.content

        # Print debug message every 50 responses
        if (idx + 1) % 50 == 0:
            print(f"✓ Processed {idx + 1}/{len(codetrans_df)} rows")

        # Optional: Add a small delay to avoid rate limiting
        time.sleep(0.1)

    except Exception as e:
        print(f"✗ Error at row {idx}: {str(e)}")
        codetrans_df.at[idx, 'cheating'] = f"ERROR: {str(e)}"
        continue

print(f"\n✓ Complete! Processed all {len(codetrans_df)} rows")

In [ ]:
import re

# Parse the 'cheating' column using regex for both dataframes
for df_name, df in [('codeocean_df', codeocean_df), ('codetrans_df', codetrans_df)]:
    print(f"\n{'='*50}")
    print(f"Processing {df_name}")
    print(f"{'='*50}")

    for idx, row in df.iterrows():
        try:
            cheating_text = str(row['cheating'])

            # Extract score: find "score:" and capture the value until comma
            score_match = re.search(r'"?score"?\s*:\s*([^,}\n]+)', cheating_text)
            if score_match:
                score_value = score_match.group(1).strip().strip('"').strip()
                # Try to convert to int/float if possible
                try:
                    score_value = int(score_value)
                except:
                    try:
                        score_value = float(score_value)
                    except:
                        pass
                df.at[idx, 'score'] = score_value
            else:
                df.at[idx, 'score'] = None

            # Extract reason: find "reason:" and capture everything from next char to last char
            reason_match = re.search(r'"?reason"?\s*:\s*"?(.+?)(?:"\s*}|\s*}|$)', cheating_text, re.DOTALL)
            if reason_match:
                reason_value = reason_match.group(1).strip().strip('"').strip()
                df.at[idx, 'reason'] = reason_value
            else:
                df.at[idx, 'reason'] = None

        except Exception as e:
            print(f"✗ Error parsing row {idx}: {str(e)}")
            df.at[idx, 'score'] = None
            df.at[idx, 'reason'] = None

    # Save the dataframe as CSV
    output_filename = f'{df_name}_processed.csv'
    df.to_csv(output_filename, index=False)
    print(f"✓ Dataframe saved as '{output_filename}'")

    # Display summary
    print(f"\nDataframe shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nSample parsed data:")
    print(df[['score', 'reason']].head())